# Prediksi Penjualan dan Clustering Toko - Data Sales Bogor

Notebook ini mengikuti alur seperti notebook referensi:
1. Import library
2. Load data
3. Cleaning data dan outlier
4. Feature engineering
5. Training model prediksi penjualan
6. Prediksi 30 hari ke depan
7. Clustering toko
8. Dashboard visualisasi
9. Export hasil

In [ ]:
# ============================================================
# 1. IMPORT LIBRARY
# ============================================================

import sys
import subprocess
import importlib.util
import warnings
warnings.filterwarnings('ignore')

required_packages = [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('seaborn', 'seaborn'),
    ('sklearn', 'scikit-learn'),
    ('openpyxl', 'openpyxl'),
]

missing = [pip_name for import_name, pip_name in required_packages if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')

In [ ]:
# ============================================================
# 2. LOAD DATA
# ============================================================

DATA_PATH = Path(r"/Users/muhghifari/Documents/KULIAH/JOKI TEMEN/nazra (pa boldson)/DATASALESBOGOR.csv")

# Jika file path lokal tidak ditemukan dan notebook dijalankan di Google Colab,
# cell ini akan meminta upload file secara manual.
if DATA_PATH.exists():
    filename = DATA_PATH
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        filename = Path(list(uploaded.keys())[0])
    except Exception as exc:
        raise FileNotFoundError(f"File tidak ditemukan: {DATA_PATH}") from exc

# Data sales Bogor memakai delimiter titik-koma (;).
df_raw = pd.read_csv(filename, sep=';', encoding='utf-8-sig')

print(f"File berhasil dimuat: {filename}")
print(f"Jumlah baris : {df_raw.shape[0]:,}")
print(f"Jumlah kolom : {df_raw.shape[1]}")
display(df_raw.head())

In [ ]:
# ============================================================
# 3. PEMBERSIHAN DATA (DATA CLEANING)
# ============================================================

# Konversi tipe data utama.
df = df_raw.copy()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
df['Qty'] = pd.to_numeric(df['Qty'], errors='coerce')

# Hapus baris kosong pada kolom penting.
required_cols = ['Date', 'Value', 'Qty', 'Brand', 'Channel', 'Kode Store', 'Nama Store', 'SKU']
df = df.dropna(subset=required_cols).copy()

# Hapus outlier ekstrem yang kemungkinan besar merupakan error input/export.
# Pada data asli ada Qty = 2,147,483,647 dan Value = 65,712,999,598,200.
outlier_mask = (df['Qty'] > 10_000) | (df['Value'] > 1_000_000_000)
outliers = df.loc[outlier_mask, ['Date', 'Nama Store', 'SKU', 'Qty', 'Value']].copy()

print(f"Outlier ekstrem terdeteksi: {len(outliers):,} baris")
if len(outliers):
    display(outliers.sort_values(['Value', 'Qty'], ascending=False))

df = df.loc[~outlier_mask].copy()
df = df.sort_values('Date').reset_index(drop=True)

print(f"Data setelah dibersihkan: {df.shape[0]:,} baris")
print(f"Rentang tanggal : {df['Date'].min().date()} s/d {df['Date'].max().date()}")
print(f"Total Qty       : {df['Qty'].sum():,.0f}")
print(f"Total Value     : Rp {df['Value'].sum():,.0f}")
print(f"Brand           : {df['Brand'].dropna().unique().tolist()}")
print(f"Channel         : {df['Channel'].dropna().unique().tolist()}")
display(df.head())

In [ ]:
# ============================================================
# 4. FEATURE ENGINEERING — PERSIAPAN FITUR PREDIKSI
# ============================================================

# Agregasi data ke level harian, lalu tambahkan fitur turunan:
# - DayNum : nomor hari sejak hari pertama
# - Month  : bulan
# - DOW    : hari dalam seminggu, 0=Senin dan 6=Minggu
# - Week   : nomor minggu dalam tahun
# - Lag1   : penjualan hari sebelumnya
# - Lag7   : penjualan tujuh hari sebelumnya
# - MA7    : rata-rata penjualan tujuh hari terakhir
# - TotalQty, TxCount, ActiveStores, ActiveSKU sebagai indikator aktivitas harian

daily = (df.groupby('Date')
           .agg(TotalValue=('Value', 'sum'),
                TotalQty=('Qty', 'sum'),
                TxCount=('Value', 'count'),
                ActiveStores=('Kode Store', 'nunique'),
                ActiveSKU=('SKU', 'nunique'))
           .reset_index()
           .sort_values('Date'))

# Lengkapi tanggal yang hilang agar lag harian tetap konsisten.
full_dates = pd.date_range(daily['Date'].min(), daily['Date'].max(), freq='D')
daily = daily.set_index('Date').reindex(full_dates).rename_axis('Date').reset_index()
for col in ['TotalValue', 'TotalQty', 'TxCount', 'ActiveStores', 'ActiveSKU']:
    daily[col] = daily[col].fillna(0)

daily['DayNum'] = (daily['Date'] - daily['Date'].min()).dt.days
daily['Month'] = daily['Date'].dt.month
daily['DOW'] = daily['Date'].dt.dayofweek
daily['Week'] = daily['Date'].dt.isocalendar().week.astype(int)
daily['IsWeekend'] = daily['DOW'].isin([5, 6]).astype(int)
daily['Lag1'] = daily['TotalValue'].shift(1)
daily['Lag7'] = daily['TotalValue'].shift(7)
daily['MA7'] = daily['TotalValue'].shift(1).rolling(7).mean()
daily['QtyLag1'] = daily['TotalQty'].shift(1)
daily['QtyMA7'] = daily['TotalQty'].shift(1).rolling(7).mean()

model_data = daily.dropna().reset_index(drop=True)

print(f"Feature engineering selesai. Total hari untuk modeling: {len(model_data):,}")
display(model_data.head())

In [ ]:
# ============================================================
# 5. PELATIHAN MODEL PREDIKSI PENJUALAN
# ============================================================

# Tiga model machine learning dibandingkan:
# 1. Linear Regression   : baseline sederhana
# 2. Random Forest       : ensemble decision tree
# 3. Gradient Boosting   : boosting model untuk pola non-linear
#
# Data dibagi 80% training dan 20% testing tanpa shuffle agar urutan waktu tetap terjaga.

features = [
    'DayNum', 'Month', 'DOW', 'Week', 'IsWeekend',
    'Lag1', 'Lag7', 'MA7', 'QtyLag1', 'QtyMA7',
    'TxCount', 'ActiveStores', 'ActiveSKU'
]

X = model_data[features]
y = model_data['TotalValue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = np.maximum(model.predict(X_test), 0)
    results[name] = {
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2': r2_score(y_test, pred),
        'pred': pred,
    }

print('Hasil evaluasi model:')
print(f"{'Model':<22} {'R2 Score':>10} {'MAE (Rp)':>16} {'RMSE (Rp)':>16}")
print('-' * 68)
for name, r in results.items():
    print(f"{name:<22} {r['R2']:>10.3f} {r['MAE']:>16,.0f} {r['RMSE']:>16,.0f}")

best_name = max(results, key=lambda n: results[n]['R2'])
best_model = models[best_name]

# Latih ulang model terbaik menggunakan seluruh data modeling sebelum forecast.
best_model.fit(X, y)
print()
print(f"Model terbaik: {best_name} (R2={results[best_name]['R2']:.3f})")

In [ ]:
# ============================================================
# 6. PREDIKSI 30 HARI KE DEPAN
# ============================================================

# Prediksi dibuat secara iteratif. Setiap prediksi hari berikutnya dipakai kembali
# sebagai Lag1/Lag7/MA7 untuk memprediksi hari setelahnya.

last_date = daily['Date'].max()
last_day = int(daily['DayNum'].max())
value_buffer = list(daily['TotalValue'].tail(7))
qty_buffer = list(daily['TotalQty'].tail(7))
activity_defaults = daily[['TxCount', 'ActiveStores', 'ActiveSKU']].tail(28).mean()

future_rows = []
for i in range(1, 31):
    fd = last_date + pd.Timedelta(days=i)
    row = {
        'DayNum': last_day + i,
        'Month': fd.month,
        'DOW': fd.dayofweek,
        'Week': int(fd.isocalendar().week),
        'IsWeekend': int(fd.dayofweek in [5, 6]),
        'Lag1': value_buffer[-1],
        'Lag7': value_buffer[-7],
        'MA7': np.mean(value_buffer[-7:]),
        'QtyLag1': qty_buffer[-1],
        'QtyMA7': np.mean(qty_buffer[-7:]),
        'TxCount': activity_defaults['TxCount'],
        'ActiveStores': activity_defaults['ActiveStores'],
        'ActiveSKU': activity_defaults['ActiveSKU'],
    }

    pred_value = max(float(best_model.predict(pd.DataFrame([row])[features])[0]), 0)

    # Estimasi quantity memakai value per quantity rata-rata 28 hari terakhir.
    recent = daily.tail(28).copy()
    recent_ratio = recent.loc[recent['TotalQty'] > 0, 'TotalValue'].sum() / max(recent.loc[recent['TotalQty'] > 0, 'TotalQty'].sum(), 1)
    pred_qty = max(pred_value / recent_ratio, 0)

    future_rows.append({
        'Date': fd,
        'Predicted_Value': round(pred_value, 0),
        'Predicted_Qty': round(pred_qty, 0),
    })

    value_buffer.append(pred_value)
    qty_buffer.append(pred_qty)

future_df = pd.DataFrame(future_rows)

print(f"Prediksi 30 hari ke depan ({future_df['Date'].min().date()} s/d {future_df['Date'].max().date()}):")
display(future_df)

In [ ]:
# ============================================================
# 7. CLUSTERING TOKO (SEGMENTASI)
# ============================================================

# K-Means digunakan untuk mengelompokkan toko berdasarkan perilaku pembelian.
# Fitur per toko:
# - TotalValue : total nilai penjualan
# - TotalQty   : total quantity
# - TxCount    : jumlah transaksi/baris penjualan
# - AvgOrder   : rata-rata nilai transaksi
# - UniqueSKU  : jumlah SKU unik
# - ActiveDays : jumlah hari toko memiliki penjualan

store_stats = df.groupby(['Kode Store', 'Nama Store', 'Channel']).agg(
    TotalValue=('Value', 'sum'),
    TotalQty=('Qty', 'sum'),
    TxCount=('Value', 'count'),
    AvgOrder=('Value', 'mean'),
    UniqueSKU=('SKU', 'nunique'),
    ActiveDays=('Date', 'nunique'),
).reset_index()

feat_clust = store_stats[['TotalValue', 'TotalQty', 'TxCount', 'AvgOrder', 'UniqueSKU', 'ActiveDays']]
scaler = StandardScaler()
X_sc = scaler.fit_transform(feat_clust)

max_k = min(7, len(store_stats) - 1)
sil_scores = {}
for k in range(2, max_k + 1):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sc)
    sil_scores[k] = silhouette_score(X_sc, labels)

best_k = max(sil_scores, key=sil_scores.get)
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
store_stats['Cluster'] = km_best.fit_predict(X_sc)

pca = PCA(n_components=2)
pcs = pca.fit_transform(X_sc)
store_stats['PC1'] = pcs[:, 0]
store_stats['PC2'] = pcs[:, 1]

cluster_profile = store_stats.groupby('Cluster').agg(
    Jumlah_Toko=('Kode Store', 'count'),
    Avg_TotalValue=('TotalValue', 'mean'),
    Avg_TotalQty=('TotalQty', 'mean'),
    Avg_TxCount=('TxCount', 'mean'),
    Avg_OrderValue=('AvgOrder', 'mean'),
    Avg_SKU=('UniqueSKU', 'mean'),
    Avg_ActiveDays=('ActiveDays', 'mean'),
).reset_index()

print(f"Clustering selesai dengan k={best_k} klaster")
print('Profil tiap klaster:')
display(cluster_profile)

In [ ]:
# ============================================================
# 8. VISUALISASI DASHBOARD LENGKAP
# ============================================================

# Dashboard berisi:
# 1. Tren historis penjualan dan prediksi 30 hari
# 2. Perbandingan R2 antar model
# 3. Perbandingan MAE antar model
# 4. Scatter plot cluster toko dengan PCA
# 5. Profil cluster
# 6. Penjualan per Channel dan Brand
# 7. Silhouette Score untuk pemilihan k

colors = ['#2563EB', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6', '#EC4899', '#14B8A6']
fig = plt.figure(figsize=(20, 22))
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.42, wspace=0.32)

# Plot 1: Historis + Forecast
ax1 = fig.add_subplot(gs[0, :])
ax1.fill_between(daily['Date'], daily['TotalValue'] / 1e6, alpha=0.18, color=colors[0])
ax1.plot(daily['Date'], daily['TotalValue'] / 1e6, color=colors[0], lw=1.5, label='Aktual')
ax1.plot(future_df['Date'], future_df['Predicted_Value'] / 1e6, color=colors[1], lw=2.5,
         linestyle='--', marker='o', markersize=4, label='Prediksi 30 Hari')
ax1.axvline(daily['Date'].max(), color='gray', linestyle=':', lw=1.5)
ax1.set_title('Tren Penjualan Harian dan Prediksi 30 Hari', fontsize=14, fontweight='bold')
ax1.set_ylabel('Total Nilai Penjualan (Juta Rp)')
ax1.legend(fontsize=10)
ax1.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))

# Plot 2: R2 Score
ax2 = fig.add_subplot(gs[1, 0])
model_names = list(results.keys())
r2_vals = [results[n]['R2'] for n in model_names]
bars = ax2.barh(model_names, r2_vals, color=[colors[0], colors[1], colors[2]])
ax2.set_title('Perbandingan Akurasi Model (R2 Score)', fontsize=12, fontweight='bold')
ax2.set_xlabel('R2 Score')
left_lim = min(0, min(r2_vals) - 0.1)
right_lim = max(1, max(r2_vals) + 0.1)
ax2.set_xlim(left_lim, right_lim)
for bar, val in zip(bars, r2_vals):
    ax2.text(val + 0.02, bar.get_y() + bar.get_height() / 2, f'{val:.3f}', va='center', fontweight='bold')

# Plot 3: MAE
ax3 = fig.add_subplot(gs[1, 1])
mae_vals = [results[n]['MAE'] / 1e6 for n in model_names]
bars2 = ax3.barh(model_names, mae_vals, color=[colors[0], colors[1], colors[2]])
ax3.set_title('MAE Model (Juta Rp) - Lebih Kecil Lebih Baik', fontsize=12, fontweight='bold')
ax3.set_xlabel('MAE (Juta Rp)')
for bar, val in zip(bars2, mae_vals):
    ax3.text(bar.get_width() + max(mae_vals) * 0.02, bar.get_y() + bar.get_height() / 2,
             f'{val:.2f}', va='center', fontweight='bold')

# Plot 4: Cluster PCA Scatter
ax4 = fig.add_subplot(gs[2, 0])
ax4.scatter(store_stats['PC1'], store_stats['PC2'],
            c=[colors[int(c) % len(colors)] for c in store_stats['Cluster']],
            alpha=0.75, edgecolors='white', linewidth=0.5, s=60)
for c in sorted(store_stats['Cluster'].unique()):
    sub = store_stats[store_stats['Cluster'] == c]
    ax4.annotate(f'Klaster {c}', (sub['PC1'].mean(), sub['PC2'].mean()),
                 fontsize=10, fontweight='bold', color=colors[int(c) % len(colors)],
                 ha='center', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
ax4.set_title(f'Clustering Toko (K-Means, k={best_k}) - PCA Visualization', fontsize=12, fontweight='bold')
ax4.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)')
ax4.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)')

# Plot 5: Cluster Profile
ax5 = fig.add_subplot(gs[2, 1])
x = np.arange(len(cluster_profile))
w = 0.35
ax5.bar(x - w/2, cluster_profile['Avg_TotalValue'] / 1e6, w, label='Avg Total Value (Juta)', color=colors[0])
ax5r = ax5.twinx()
ax5r.bar(x + w/2, cluster_profile['Avg_TxCount'], w, label='Avg Tx Count', color=colors[1])
ax5.set_xticks(x)
ax5.set_xticklabels([f"Klaster {int(i)}" for i in cluster_profile['Cluster']])
ax5.set_title('Profil Setiap Klaster', fontsize=12, fontweight='bold')
ax5.set_ylabel('Avg Total Value (Juta Rp)', color=colors[0])
ax5r.set_ylabel('Avg Tx Count', color=colors[1])
lines = [plt.Line2D([0], [0], color=colors[0], lw=4), plt.Line2D([0], [0], color=colors[1], lw=4)]
ax5.legend(lines, ['Avg Total Value (Juta)', 'Avg Tx Count'], loc='upper right', fontsize=8)

# Plot 6: Sales by Channel & Brand
ax6 = fig.add_subplot(gs[3, 0])
ch_brand = df.groupby(['Channel', 'Brand'])['Value'].sum().unstack().fillna(0) / 1e6
ch_brand.plot(kind='bar', ax=ax6, color=colors[:len(ch_brand.columns)], edgecolor='white')
ax6.set_title('Nilai Penjualan per Channel dan Brand (Juta Rp)', fontsize=12, fontweight='bold')
ax6.set_xlabel('Channel')
ax6.set_ylabel('Total Nilai (Juta Rp)')
ax6.legend(title='Brand', fontsize=8)
ax6.tick_params(axis='x', rotation=0)

# Plot 7: Silhouette Score
ax7 = fig.add_subplot(gs[3, 1])
ax7.plot(list(sil_scores.keys()), list(sil_scores.values()), marker='o', color=colors[2], lw=2.5, markersize=8)
ax7.axvline(best_k, color=colors[3], linestyle='--', lw=2, label=f'Best k={best_k}')
ax7.set_title('Silhouette Score untuk Menentukan Jumlah Klaster', fontsize=12, fontweight='bold')
ax7.set_xlabel('Jumlah Klaster (k)')
ax7.set_ylabel('Silhouette Score')
ax7.legend(fontsize=10)
ax7.set_xticks(list(sil_scores.keys()))

fig.suptitle('Sales Analytics Dashboard - Bogor Region 2025\nPrediksi Penjualan dan Segmentasi Toko',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('sales_analysis_bogor.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("Dashboard tersimpan sebagai 'sales_analysis_bogor.png'")

In [ ]:
# ============================================================
# 9. EKSPOR HASIL KE EXCEL DAN CSV
# ============================================================

# File yang dibuat:
# - sales_forecast_30_days.csv
# - clustering_toko_bogor.xlsx
# - model_evaluation_bogor.csv
# - sales_analysis_bogor.png dari cell dashboard

forecast_out = future_df.copy()
forecast_out.to_csv('sales_forecast_30_days.csv', index=False)

model_eval = pd.DataFrame([
    {'Model': name, 'MAE': r['MAE'], 'RMSE': r['RMSE'], 'R2': r['R2']}
    for name, r in results.items()
]).sort_values('R2', ascending=False)
model_eval.to_csv('model_evaluation_bogor.csv', index=False)

cluster_out = store_stats[[
    'Kode Store', 'Nama Store', 'Channel',
    'TotalValue', 'TotalQty', 'TxCount', 'AvgOrder', 'UniqueSKU', 'ActiveDays', 'Cluster'
]].copy()
cluster_out.columns = [
    'Kode Store', 'Nama Store', 'Channel',
    'Total Value (Rp)', 'Total Qty', 'Jumlah Transaksi',
    'Avg Order (Rp)', 'Jumlah SKU', 'Jumlah Hari Aktif', 'Klaster'
]
cluster_out.to_excel('clustering_toko_bogor.xlsx', index=False)

print('Export selesai:')
print('- sales_forecast_30_days.csv')
print('- clustering_toko_bogor.xlsx')
print('- model_evaluation_bogor.csv')
print('- sales_analysis_bogor.png')